# 05D — YOLO Tumor Detector

Trains a YOLOv8n one-class tumor detector. Run this in a **separate session** from 05B to avoid disk conflicts.

### Datasets to attach
| Dataset | Role |
|---|---|
| Your code zip (`brats-seg-code`) | Source code |
| `brats2020-2d-prepared` (05A output) | Prepared slices (read-only, doesn't cost disk) |

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
CODE_DATASET_SLUG = 'brats-seg-code'
YOLO_EPOCHS       = 50
YOLO_IMAGE_SIZE   = 240
MARGIN            = 8   # pixels added around tumor bbox

In [ ]:
import os, shutil, subprocess, sys, yaml
from pathlib import Path

def disk_gb(path='/'):
    st = os.statvfs(path)
    return round(st.f_bavail * st.f_frsize / 1e9, 2)

print(f'Free disk: {disk_gb()} GB')

In [ ]:
# ── 1. Copy repo ───────────────────────────────────────────────────────────
def looks_like_repo(p):
    return (p / 'src').exists() and (p / 'scripts').exists() and (p / 'configs').exists()

repo_input = Path('/kaggle/input') / CODE_DATASET_SLUG
if not looks_like_repo(repo_input):
    for p in Path('/kaggle/input').rglob('requirements.txt'):
        if looks_like_repo(p.parent):
            repo_input = p.parent; break

PROJECT = Path('/kaggle/working/project')
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(repo_input, PROJECT,
    ignore=shutil.ignore_patterns('.git', '__pycache__', '.pytest_cache', 'data', 'outputs', 'checkpoints'))
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

def run(cmd):
    subprocess.check_call(cmd, cwd=str(PROJECT))

print('Repo:', repo_input)

In [ ]:
# ── 2. Install packages ────────────────────────────────────────────────────
# ultralytics may already be on Kaggle but version can be stale
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'albumentations'])
import torch
print(f'CUDA: {torch.cuda.is_available()}  |  Free disk: {disk_gb()} GB')

In [ ]:
# ── 3. Find prepared data ──────────────────────────────────────────────────
PREPARED = next(
    (p for p in [
        Path('/kaggle/input/brats2020-2d-prepared'),
        Path('/kaggle/input/brats2020_2d_prepared'),
    ] if (p / 'data/brats2020_2d/metadata.json').exists()),
    None
)
if PREPARED is None:
    raise FileNotFoundError('Attach the brats2020-2d-prepared dataset (output of 05A).')

DATA_ROOT  = PREPARED / 'data/brats2020_2d'
SPLIT_FILE = PREPARED / 'configs/splits/default.json'
print(f'Prepared data: {DATA_ROOT}')

In [ ]:
# ── 4. Runtime config ──────────────────────────────────────────────────────
with open(PROJECT / 'configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['data_root']  = str(DATA_ROOT)
cfg['data']['split_file'] = str(SPLIT_FILE)
cfg['checkpoint_dir']     = '/kaggle/working/checkpoints'
cfg['experiment']['output_dir'] = '/kaggle/working/outputs'
rt = Path('/kaggle/working/rt_yolo.yaml')
with open(rt, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('Runtime config written.')

In [ ]:
# ── 5. Build YOLO image/label dataset ──────────────────────────────────────
YOLO_DATA = Path('/kaggle/working/yolo_tumor')

if not (YOLO_DATA / 'data.yaml').exists():
    print('Building YOLO dataset from prepared slices …')
    run([
        sys.executable, '-m', 'scripts.prepare_yolo',
        '--config', str(rt),
        '--output', str(YOLO_DATA),
        '--margin', str(MARGIN),
    ])
else:
    print('YOLO dataset already exists.')

def folder_gb(path):
    return round(sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file()) / 1e9, 2)

print(f'YOLO dataset size: {folder_gb(YOLO_DATA)} GB')
print(f'Free disk: {disk_gb()} GB')

In [ ]:
# ── 6. Train YOLO ──────────────────────────────────────────────────────────
# YOLO best.pt lands in /kaggle/working/outputs/yolo/tumor_detector/weights/
YOLO_PROJECT = '/kaggle/working/outputs/yolo'
YOLO_BEST    = Path(YOLO_PROJECT) / 'tumor_detector/weights/best.pt'

if not YOLO_BEST.exists():
    run([
        sys.executable, '-m', 'scripts.train_yolo',
        '--data-yaml', str(YOLO_DATA / 'data.yaml'),
        '--model',     'yolov8n.pt',
        '--epochs',    str(YOLO_EPOCHS),
        '--imgsz',     str(YOLO_IMAGE_SIZE),
        '--project',   YOLO_PROJECT,
    ])
else:
    print('YOLO checkpoint already exists:', YOLO_BEST)

print(f'Free disk: {disk_gb()} GB')

In [ ]:
# ── 7. Quick validation + sample detections ────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from ultralytics import YOLO
from src.data.brats import BraTSDataset
from src.data.splits import load_split
from src.data.yolo import modality_rgb

detector = YOLO(str(YOLO_BEST))
split = load_split(str(SPLIT_FILE))
ds = BraTSDataset(str(DATA_ROOT), patient_ids=split['test'])

# Find 4 test slices with tumor
samples = []
for img, msk in ds:
    if (msk > 0).float().mean() > 0.03 and len(samples) < 4:
        samples.append((img.numpy(), msk.numpy()))

fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 4))
for ax, (img_np, msk_np) in zip(axes, samples):
    rgb = modality_rgb(img_np)  # (H,W,3) uint8
    results = detector(rgb, verbose=False)[0]
    ax.imshow(rgb)
    for box in results.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                       linewidth=2, edgecolor='cyan', facecolor='none'))
    ax.axis('off')
plt.suptitle('YOLOv8n — Test detections (cyan = predicted box)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Package YOLO artifacts ──────────────────────────────────────────────
EXPORT = Path('/kaggle/working/yolo_artifacts')
if EXPORT.exists(): shutil.rmtree(EXPORT)
EXPORT.mkdir()
yolo_out = Path('/kaggle/working/outputs/yolo')
if yolo_out.exists():
    shutil.copytree(yolo_out, EXPORT / 'yolo')
archive = shutil.make_archive(str(EXPORT), 'zip', EXPORT)
print('Packaged:', archive)
print(f'Free disk: {disk_gb()} GB')